# Project Lab 1: Pipelines, Lakes, Batches, Loaders and Models

Notebook for **Chapter 15** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to Run a Project in This Area

## 15.2 Project: API to Scheduler to Database Pipeline in Containers

In [ ]:
import sqlite3, time, random

class FakeAPI:
    def __init__(self): self.calls = 0
    def get_deliveries(self, day):
        self.calls += 1
        if self.calls <= 2:
            raise ConnectionError("503 service unavailable")
        rnd = random.Random(day)
        return [{"delivery_id": f"{day}-{i}", "city": rnd.choice(["Pune", "Delhi", "Chennai"]),
                 "minutes": rnd.randint(20, 90)} for i in range(5)]

def with_retry(fn, attempts=4, base_delay=0.01):
    for n in range(1, attempts + 1):
        try:
            return fn()
        except ConnectionError as e:
            print(f"  attempt {n} failed: {e}")
            if n == attempts: raise
            time.sleep(base_delay * 2 ** (n - 1))

def load(db, rows):
    db.executemany("""INSERT INTO deliveries(delivery_id, city, minutes)
        VALUES(:delivery_id,:city,:minutes)
        ON CONFLICT(delivery_id) DO UPDATE SET city=excluded.city, minutes=excluded.minutes""", rows)
    db.commit()

def run_scheduler(db, api, days):
    for day in days:
        if db.execute("SELECT 1 FROM runs WHERE day=? AND status='ok'", (day,)).fetchone():
            print(f"{day}: already done, skip"); continue
        print(f"{day}: running")
        try:
            rows = with_retry(lambda: api.get_deliveries(day))
            load(db, rows)
            db.execute("INSERT OR REPLACE INTO runs VALUES(?,?,?)", (day, "ok", len(rows)))
        except Exception:
            db.execute("INSERT OR REPLACE INTO runs VALUES(?,?,?)", (day, "failed", 0))
        db.commit()

db = sqlite3.connect(":memory:")
db.executescript("""CREATE TABLE deliveries(delivery_id TEXT PRIMARY KEY, city TEXT, minutes INT);
                    CREATE TABLE runs(day TEXT PRIMARY KEY, status TEXT, rows INT);""")
api = FakeAPI(); days = ["2024-06-01", "2024-06-02"]
run_scheduler(db, api, days)
print("--- run again (idempotent) ---")
run_scheduler(db, api, days)
print(db.execute("SELECT COUNT(*) FROM deliveries").fetchone(), db.execute("SELECT * FROM runs").fetchall())

In [ ]:
# dags/deliveries.py  (not run here)
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

with DAG("deliveries", start_date=datetime(2024, 6, 1), schedule="@daily", catchup=True,
         default_args={"retries": 3, "retry_delay": timedelta(minutes=2)}) as dag:
    PythonOperator(task_id="extract_load", python_callable=lambda ds: print("run for", ds))

## 15.3 Project: End-to-End Cloud-Style Pipeline

In [ ]:
import csv, json, random, sqlite3, shutil, pathlib
root = pathlib.Path("/tmp/lab_de/lake"); shutil.rmtree(root, ignore_errors=True)
for z in ("raw", "clean", "report"): (root / z).mkdir(parents=True)

rnd = random.Random(7); rows = []
for i in range(1, 21):
    rows.append({"order_id": f"O{i}", "customer": rnd.choice(["asha","RAVI","Meena"]),
                 "amount": rnd.choice([250, 480, 120, 99, -5, ""]), "date": f"2024-06-0{rnd.randint(1,3)}"})
rows += rows[:3]
with open(root/"raw/orders.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)

seen, clean, rejected = set(), [], 0
for r in csv.DictReader(open(root/"raw/orders.csv")):
    if r["order_id"] in seen or r["amount"] == "" or float(r["amount"]) < 0:
        rejected += 1; continue
    seen.add(r["order_id"])
    clean.append({"order_id": r["order_id"], "customer": r["customer"].title(),
                  "amount": float(r["amount"]), "date": r["date"]})
json.dump(clean, open(root/"clean/orders.json", "w"))
print("raw rows", len(rows), "clean rows", len(clean), "rejected", rejected)

db = sqlite3.connect(root/"warehouse.db")
db.executescript("""CREATE TABLE dim_customer(customer_key INTEGER PRIMARY KEY, name TEXT UNIQUE);
  CREATE TABLE fact_orders(order_id TEXT PRIMARY KEY, customer_key INT, amount REAL, date TEXT);""")
for r in clean:
    db.execute("INSERT OR IGNORE INTO dim_customer(name) VALUES(?)", (r["customer"],))
    key = db.execute("SELECT customer_key FROM dim_customer WHERE name=?", (r["customer"],)).fetchone()[0]
    db.execute("INSERT INTO fact_orders VALUES(?,?,?,?)", (r["order_id"], key, r["amount"], r["date"]))
db.commit()
res = db.execute("""SELECT c.name, COUNT(*), SUM(f.amount) FROM fact_orders f
    JOIN dim_customer c USING(customer_key) GROUP BY c.name ORDER BY 3 DESC""").fetchall()
with open(root/"report/revenue_by_customer.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["customer","orders","revenue"]); w.writerows(res)
print(open(root/"report/revenue_by_customer.csv").read())
assert sum(r[2] for r in res) == sum(r["amount"] for r in clean)
print("reconciled:", sum(r["amount"] for r in clean))

## 15.4 Project: Large Batch Processing with Spark-Style Partitioning

In [ ]:
import random, time, zlib
from multiprocessing import Pool
from collections import Counter
N, P = 1_200_000, 8
def make_rows(n):
    rnd = random.Random(1); cities = ["Pune","Delhi","Chennai","Mumbai","Kolkata"]
    return [(rnd.choice(cities), rnd.randint(20, 120)) for _ in range(n)]
def partition(rows, p):
    parts = [[] for _ in range(p)]
    for city, m in rows: parts[zlib.crc32(city.encode()) % p].append((city, m))
    return parts
def map_task(part):
    s, c = Counter(), Counter()
    for city, m in part: s[city] += m; c[city] += 1
    return s, c
def reduce_all(results):
    S, C = Counter(), Counter()
    for s, c in results: S.update(s); C.update(c)
    return {k: round(S[k] / C[k], 2) for k in sorted(S)}
if __name__ == "__main__":
    rows = make_rows(N); parts = partition(rows, P)
    print("partition sizes:", [len(x) for x in parts])
    t = time.perf_counter(); serial = reduce_all([map_task(rows)]); t1 = time.perf_counter() - t
    t = time.perf_counter()
    with Pool(4) as pool: par = reduce_all(pool.map(map_task, parts))
    t2 = time.perf_counter() - t
    print("serial  ", serial); print("parallel", par)
    assert serial == par
    print(f"results equal; serial {t1:.2f}s, 4 workers {t2:.2f}s")
    skew = [("Pune", 50)] * 900_000 + rows[:100_000]
    print("skewed partition sizes:", [len(x) for x in partition(skew, P)])

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("deliveries").getOrCreate()
df = spark.read.parquet("deliveries/")
out = df.repartition(8, "city").groupBy("city").agg(F.avg("minutes").alias("avg_min"))
out.write.mode("overwrite").parquet("out/avg_by_city")

## 15.5 Project: Incremental ETL Loader with Watermarks and Tests

In [ ]:
import sqlite3, unittest

def setup():
    db = sqlite3.connect(":memory:")
    db.executescript("""
    CREATE TABLE src_orders(order_id TEXT PRIMARY KEY, amount REAL, updated_at TEXT);
    CREATE TABLE tgt_orders(order_id TEXT PRIMARY KEY, amount REAL, updated_at TEXT);
    CREATE TABLE watermark(name TEXT PRIMARY KEY, value TEXT);
    INSERT INTO watermark VALUES('orders','1970-01-01T00:00:00');""")
    return db

def incremental_load(db, overlap_minutes=0):
    wm = db.execute("SELECT value FROM watermark WHERE name='orders'").fetchone()[0]
    rows = db.execute("""SELECT order_id, amount, updated_at FROM src_orders
        WHERE updated_at > strftime('%Y-%m-%dT%H:%M:%S', ?, ?)""",
        (wm, f"-{overlap_minutes} minutes")).fetchall()
    if not rows: return 0
    db.executemany("""INSERT INTO tgt_orders VALUES(?,?,?) ON CONFLICT(order_id)
        DO UPDATE SET amount=excluded.amount, updated_at=excluded.updated_at""", rows)
    db.execute("UPDATE watermark SET value=? WHERE name='orders'", (max(r[2] for r in rows),))
    db.commit()
    return len(rows)

class T(unittest.TestCase):
    def test_second_run_loads_nothing(self):
        db = setup()
        db.execute("INSERT INTO src_orders VALUES('O1',100,'2024-06-01T10:00:00')")
        incremental_load(db)
        self.assertEqual(incremental_load(db), 0)
    def test_update_is_picked_up_not_duplicated(self):
        db = setup()
        db.execute("INSERT INTO src_orders VALUES('O1',100,'2024-06-01T10:00:00')")
        incremental_load(db)
        db.execute("UPDATE src_orders SET amount=150, updated_at='2024-06-02T09:00:00' WHERE order_id='O1'")
        incremental_load(db)
        self.assertEqual(db.execute("SELECT COUNT(*), MAX(amount) FROM tgt_orders").fetchone(), (1, 150.0))
    def test_late_row_is_lost_without_overlap(self):
        db = setup()
        db.execute("INSERT INTO src_orders VALUES('O1',100,'2024-06-01T10:00:00')")
        incremental_load(db)
        db.execute("INSERT INTO src_orders VALUES('O2',50,'2024-06-01T09:55:00')")  # committed late
        self.assertEqual(incremental_load(db, overlap_minutes=0), 0)    # lost!
        self.assertEqual(incremental_load(db, overlap_minutes=10), 2)   # overlap rescues it
        self.assertEqual(db.execute("SELECT COUNT(*) FROM tgt_orders").fetchone()[0], 2)
if __name__ == "__main__":
    unittest.main(verbosity=2)

## 15.6 Project: dbt-Style Project, Staging to Marts with Tests and Docs

In [ ]:
import re, sqlite3
from graphlib import TopologicalSorter
MODELS = {
 "stg_orders": "SELECT order_id, customer_id, CAST(amount AS REAL) AS amount, LOWER(status) AS status FROM raw_orders",
 "stg_customers": "SELECT customer_id, UPPER(city) AS city FROM raw_customers",
 "fct_orders": "SELECT o.order_id, o.customer_id, c.city, o.amount FROM {{ ref('stg_orders') }} o JOIN {{ ref('stg_customers') }} c USING(customer_id) WHERE o.status = 'paid'",
 "mart_city_revenue": "SELECT city, COUNT(*) AS orders, SUM(amount) AS revenue FROM {{ ref('fct_orders') }} GROUP BY city",
}
TESTS = [("stg_orders","order_id","unique"), ("stg_orders","order_id","not_null"),
         ("stg_customers","customer_id","unique"), ("fct_orders","customer_id","not_null"),
         ("mart_city_revenue","city","unique")]
DOCS = {"mart_city_revenue": "Paid revenue and order count per city. One row per city."}
deps = lambda sql: set(re.findall(r"ref\('(\w+)'\)", sql))
render = lambda sql: re.sub(r"\{\{ ref\('(\w+)'\) \}\}", r"\1", sql)

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE raw_orders(order_id TEXT, customer_id TEXT, amount TEXT, status TEXT);
INSERT INTO raw_orders VALUES('O1','C1','250','PAID'),('O2','C2','480','Paid'),('O3','C1','99','CANCELLED'),('O4','C3','120','PAID'),('O4','C3','120','PAID');
CREATE TABLE raw_customers(customer_id TEXT, city TEXT);
INSERT INTO raw_customers VALUES('C1','pune'),('C2','delhi'),('C3','pune');""")
order = list(TopologicalSorter({m: deps(s) for m, s in MODELS.items()}).static_order())
print("build order:", order)
for m in order:
    db.execute(f"DROP VIEW IF EXISTS {m}"); db.execute(f"CREATE VIEW {m} AS {render(MODELS[m])}")
def run_test(model, col, kind):
    sql = {"unique": f"SELECT COUNT(*) FROM (SELECT {col} FROM {model} GROUP BY {col} HAVING COUNT(*)>1)",
           "not_null": f"SELECT COUNT(*) FROM {model} WHERE {col} IS NULL"}[kind]
    return db.execute(sql).fetchone()[0]
for t in TESTS:
    bad = run_test(*t)
    print(f"  {'PASS' if bad == 0 else 'FAIL'} {t[2]}({t[0]}.{t[1]}) bad={bad}")
print("mart:", db.execute("SELECT * FROM mart_city_revenue ORDER BY city").fetchall())

## 15.7 Break It on Purpose: Three Drills and a Check Set

A pipeline you have never seen fail is a pipeline you do not understand. Each drill below breaks one part of Project Lab 1 on purpose, shows the symptom with real output, and gives you the check that would have caught it. Do them on your own copy of the lab, then add the check to your test suite. Each drill also prints a **fingerprint** (a short hash of the table contents): put the expected fingerprint in your README, and anyone who reruns the project can tell in one second whether they got the same result.

In [ ]:
# Drill 1: remove the upsert on purpose and watch the rerun duplicate rows.
import sqlite3, hashlib
rows = [(1, "Pune", 30), (2, "Delhi", 41), (3, "Pune", 28)]
def make(upsert):
    db = sqlite3.connect(":memory:")
    db.execute("CREATE TABLE d(id INTEGER, city TEXT, minutes INTEGER)" if not upsert else
               "CREATE TABLE d(id INTEGER PRIMARY KEY, city TEXT, minutes INTEGER)")
    return db
def load(db, upsert):
    if upsert:
        db.executemany("INSERT INTO d VALUES (?,?,?) ON CONFLICT(id) DO UPDATE SET city=excluded.city, minutes=excluded.minutes", rows)
    else:
        db.executemany("INSERT INTO d VALUES (?,?,?)", rows)
    db.commit()
for upsert in (False, True):
    db = make(upsert)
    load(db, upsert); load(db, upsert)               # run twice
    n = db.execute("SELECT COUNT(*) FROM d").fetchone()[0]
    digest = hashlib.md5(repr(db.execute("SELECT * FROM d ORDER BY id, minutes").fetchall()).encode()).hexdigest()[:8]
    print("upsert" if upsert else "plain insert", "-> rows after two runs:", n, "| table fingerprint:", digest)

plain insert -> rows after two runs: 6 | table fingerprint: b991013e
upsert -> rows after two runs: 3 | table fingerprint: 14d4c622


In [ ]:
# Drill 2: an INNER JOIN silently drops rows; a reconciliation check exposes it. Then fix it.
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE orders(order_id INTEGER, customer_id TEXT, amount REAL);
CREATE TABLE customers(customer_id TEXT, city TEXT);
INSERT INTO orders VALUES (1,'c1',100),(2,'c2',50),(3,'c9',70),(4,NULL,20);
INSERT INTO customers VALUES ('c1','Pune'),('c2','Delhi');
""")
src = db.execute("SELECT COUNT(*), SUM(amount) FROM orders").fetchone()
inner = db.execute("SELECT COUNT(*), SUM(o.amount) FROM orders o JOIN customers c USING(customer_id)").fetchone()
left = db.execute("SELECT COUNT(*), SUM(o.amount), SUM(c.city IS NULL) FROM orders o LEFT JOIN customers c USING(customer_id)").fetchone()
print("source        :", src)
print("inner join    :", inner, " -> rows lost:", src[0] - inner[0], " revenue lost:", src[1] - inner[1])
print("left join     :", left[:2], " unmatched rows:", left[2], "(send these to quarantine)")
assert left[:2] == src

source        : (4, 240.0)
inner join    : (2, 150.0)  -> rows lost: 2  revenue lost: 90.0
left join     : (4, 240.0)  unmatched rows: 2 (send these to quarantine)


In [ ]:
# Drill 3: partitioning by a skewed key, then by a better one.
import zlib, collections
events = [("IN", i) for i in range(9000)] + [("DE", i) for i in range(600)] + [("US", i) for i in range(400)]
def sizes(keyfn, p=4):
    c = collections.Counter(zlib.crc32(str(keyfn(e)).encode()) % p for e in events)
    return [c.get(i, 0) for i in range(p)]
print("by country:", sizes(lambda e: e[0]))
print("by event id:", sizes(lambda e: e[1]))
print("by (country, id // 1000):", sizes(lambda e: (e[0], e[1] // 1000)))

by country: [9400, 600, 0, 0]
by event id: [2497, 2503, 2497, 2503]
by (country, id // 1000): [2000, 3000, 3000, 2000]


## Chapter recap

## Practice set

**1.** (Easy) Why does the scheduler check the `runs` table before extracting? **Answer:** To skip days that already succeeded, so a rerun does no extra work and creates no duplicates.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch15.html